# SQL-First RAG Support Chatbot

A customer support chatbot for a small business website:

1. **Scrape** the site and save each page as a PDF
2. **Generate Q&A pairs** with Claude and store them in **SQLite**
3. **Embed** the pages into **Pinecone**
4. **Route** each question: stored Q&A first → page search → Claude → polite fallback
5. **Chat** through a Gradio interface

Fill in the **Settings** cell first, then run the cells in order. See the README for design decisions and evaluation results.

## Settings

In [ ]:
# ===== SETTINGS: point the bot at any small business website =====
SITE_NAME = "Example Yoga Studio"
SITE_DESCRIPTION = "a yoga and Pilates studio"
BASE_URL = "https://www.example-yoga-studio.com/"   # check the site's robots.txt before scraping
CONTACT_PHONE = "<studio phone number>"
SCHEDULE_URL = BASE_URL + "schedule/"

# Pages to scrape, relative to BASE_URL (hand-pick these from the site's menu)
PAGE_PATHS = [
    "",            # home page
    "classes/",
    "prices/",
    "faq/",
    "about/",
    "contact/",
]

INDEX_NAME = "studio-support"   # Pinecone index for this project

## Step 0: Install, load keys, test connections

Add your keys in Colab's **Secrets** panel (🔑) as `Anthropic_API_Key` and `Pinecone_API_Key`, with notebook access switched on.

In [ ]:
!pip install -qU langchain-anthropic pinecone langchain-pinecone langchain-huggingface langchain-text-splitters langchain-community pypdf beautifulsoup4 fpdf2

import os
from google.colab import userdata

# Load keys from Colab Secrets
os.environ["ANTHROPIC_API_KEY"] = userdata.get("Anthropic_API_Key")
os.environ["PINECONE_API_KEY"] = userdata.get("Pinecone_API_Key")

# Test Claude
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-sonnet-5-5")
print(llm.invoke("Reply with just: Claude is connected").content)

# Test Pinecone
from pinecone import Pinecone
pc = Pinecone()
print("Pinecone indexes:", pc.list_indexes().names())

## Step 1: Scrape the pages and save them as PDFs

Fetches each page politely (1 second apart), strips menus, footers and scripts, removes duplicated blocks, and saves the readable text as a PDF.

In [ ]:
import time, requests
from bs4 import BeautifulSoup
from fpdf import FPDF

URLS = [BASE_URL + path for path in PAGE_PATHS]
HEADERS = {"User-Agent": "Mozilla/5.0 (student RAG course project)"}
os.makedirs("pages", exist_ok=True)

def get_page_text(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    # Remove parts that aren't content: menus, footer, scripts, forms
    for tag in soup(["script", "style", "nav", "header", "footer", "noscript", "form"]):
        tag.decompose()

    title = soup.title.get_text(strip=True) if soup.title else url
    lines = [line.strip() for line in soup.get_text(separator="\n").splitlines()]
    lines = [line for line in lines if line]

    # Many themes repeat text (twice in a row, or whole paragraphs again later)
    clean, seen = [], set()
    for line in lines:
        key = line.lower()
        if clean and key == clean[-1].lower():
            continue                  # repeated heading right after itself
        if len(line) > 40 and key in seen:
            continue                  # long paragraph we've already kept
        seen.add(key)
        clean.append(line)
    return title, "\n".join(clean)

# The built-in PDF font only supports basic characters, so swap curly quotes etc.
def to_basic_chars(text):
    swaps = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
             "\u2013": "-", "\u2014": "-", "\u2026": "...", "\u00a0": " ", "\u2022": "-"}
    for old, new in swaps.items():
        text = text.replace(old, new)
    return text.encode("latin-1", "replace").decode("latin-1")

def save_pdf(title, url, text, path):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.multi_cell(0, 8, to_basic_chars(title), new_x="LMARGIN", new_y="NEXT")
    pdf.set_font("Helvetica", "I", 9)
    pdf.multi_cell(0, 6, f"Source: {url}", new_x="LMARGIN", new_y="NEXT")
    pdf.ln(4)
    pdf.set_font("Helvetica", "", 11)
    pdf.multi_cell(0, 6, to_basic_chars(text), new_x="LMARGIN", new_y="NEXT")
    pdf.output(path)

pages = []  # keep the text in memory too - Step 2 uses it
for url, path in zip(URLS, PAGE_PATHS):
    try:
        title, text = get_page_text(url)
        slug = path.strip("/").replace("/", "-") or "home"
        pdf_path = f"pages/{slug}.pdf"
        save_pdf(title, url, text, pdf_path)
        pages.append({"url": url, "title": title, "text": text, "pdf": pdf_path})
        print(f"✅ {slug:28} {len(text):6} chars")
    except Exception as e:
        print(f"❌ {url}: {e}")
    time.sleep(1)  # be polite to their server

print(f"\nSaved {len(pages)} PDFs in the 'pages' folder")

## Step 2a: Generate Q&A pairs with Claude

Claude reads the whole site once and returns Q&A pairs in a fixed shape (structured output). Haiku keeps this one-off step cheap. Review the output before saving it.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class QAPair(BaseModel):
    question: str = Field(description="A question a real customer might ask")
    answer: str = Field(description="Short answer using only facts from the website text")
    source_url: str = Field(description="URL of the page the answer comes from")

class QAList(BaseModel):
    pairs: List[QAPair]

site_text = "\n\n".join(f"=== PAGE: {p['url']} ===\n{p['text']}" for p in pages)
print(f"Sending {len(site_text):,} characters to Claude...")

qa_llm = ChatAnthropic(model="claude-haiku-5-5", max_tokens=8000).with_structured_output(QAList)

qa_prompt = f"""You are helping build a customer support chatbot for {SITE_NAME},
{SITE_DESCRIPTION}.

Below is the text of their website. Write exactly 20 question-and-answer pairs
that a real customer would be likely to ask.

Rules:
- Base every answer ONLY on the website text. Do not add outside facts.
- Phrase questions the way a customer would: simple, everyday wording.
- Keep answers short: 1-3 sentences, friendly tone, written as "we/our".
- Cover a spread of topics: prices, location and opening hours, beginners,
  class types, facilities, booking and cancellations.
- Don't quote class times or one-off dates; point to {SCHEDULE_URL} instead.
- For each pair, give the URL of the page the answer came from.
- Prioritise questions from the FAQ page ({BASE_URL}faq/), since those
  reflect what customers actually ask. Use other pages only to fill gaps.

WEBSITE TEXT:
{site_text}
"""

result = qa_llm.invoke(qa_prompt)
qa_pairs = result.pairs

print(f"Got {len(qa_pairs)} Q&A pairs\n")
for i, qa in enumerate(qa_pairs, 1):
    print(f"{i}. Q: {qa.question}\n   A: {qa.answer}\n   ({qa.source_url})\n")

## Step 2b: Store the Q&A pairs in SQLite

⚠️ This drops and recreates the table, so only rerun it straight after Step 2a.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("support.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS faq")
cur.execute("""
    CREATE TABLE faq (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        question    TEXT NOT NULL,
        answer      TEXT NOT NULL,
        source_url  TEXT
    )
""")

cur.executemany(
    "INSERT INTO faq (question, answer, source_url) VALUES (?, ?, ?)",
    [(qa.question, qa.answer, qa.source_url) for qa in qa_pairs]
)
conn.commit()

pd.read_sql("SELECT id, question, answer FROM faq", conn)

## Step 3: Chunk the PDFs and store them in Pinecone

Chunks carry their source URL (so answers can cite pages) and fixed IDs (so reruns overwrite instead of duplicating).

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec

docs = []
for p in pages:
    loaded = PyPDFLoader(p["pdf"]).load()
    for d in loaded:
        d.metadata = {"source": p["url"], "title": p["title"]}
    docs.extend(loaded)
print(f"Loaded {len(docs)} PDF pages from {len(pages)} web pages")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",   # 768-dimensional embeddings
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=768,          # must match the embedding model
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(2)
    print(f"Created index '{INDEX_NAME}'")

vectorstore = PineconeVectorStore(index=pc.Index(INDEX_NAME), embedding=embeddings)
ids = [f"chunk-{i}" for i in range(len(chunks))]
vectorstore.add_documents(chunks, ids=ids)
print(f"Uploaded {len(chunks)} chunks to Pinecone ✅")

### Test retrieval on its own (no LLM)

Checks that the right pages are found before Claude is involved. Compare the scores of on-topic and off-topic questions: the gap informs the thresholds in Step 4.

In [ ]:
for q in ["Can I bring my baby to a class?",
          "How much does a class cost?",
          "Is there somewhere to sweat after class?",
          "Do you sell protein shakes?"]:
    print(f"\nQ: {q}")
    for doc, score in vectorstore.similarity_search_with_score(q, k=3):
        print(f"  {score:.3f}  {doc.metadata['source']}")

## Step 4a: Load the Q&A from SQL and calibrate the matcher

Each customer question is compared with every stored question. Check which rephrasings match correctly and at what score, then set `QA_THRESHOLD` in Step 4b just above the highest wrong match.

In [ ]:
import numpy as np

conn = sqlite3.connect("support.db")

def load_faq():
    global faq_rows, faq_vectors
    faq_rows = conn.execute("SELECT id, question, answer, source_url FROM faq").fetchall()
    faq_vectors = np.array(embeddings.embed_documents([row[1] for row in faq_rows]))
    print(f"Loaded {len(faq_rows)} Q&A pairs")

def best_faq_match(question):
    q_vec = np.array(embeddings.embed_query(question))
    scores = faq_vectors @ q_vec          # cosine similarity with every stored question
    best = int(np.argmax(scores))
    return faq_rows[best], float(scores[best])

load_faq()

tests = [
    "how much is a class",
    "where are you based",
    "when is reception open on a saturday",
    "im a total beginner can i come",
    "do you sell protein shakes",
    "can i park my car outside",
]
for q in tests:
    row, score = best_faq_match(q)
    print(f"{score:.3f}  '{q}'\n       → matched: {row[1]}\n")

### Review and improve the stored answers

The SQL filter finds **candidates** (answers quoting times, starting with "Yes", or using the business name in the third person); a human decides what actually needs changing. Fill gaps with new pairs and fix wording with `UPDATE`, always using the website's own facts.

In [ ]:
# 1. Find answers worth reviewing
rows = conn.execute("""
    SELECT id, question, answer FROM faq
    WHERE answer LIKE '%pm%' OR answer LIKE '%am %'
       OR answer LIKE 'Yes%' OR answer LIKE ?
""", (f"%{SITE_NAME}%",)).fetchall()
for id_, question, answer_text in rows:
    print(f"[{id_}] {question}\n{answer_text}\n")

# 2. Add pairs for common questions the generated list missed
#    e.g. ("Where are you located?", "<address from the website>", BASE_URL + "contact/")
new_pairs = []
for q, a, src in new_pairs:
    if not conn.execute("SELECT 1 FROM faq WHERE question = ?", (q,)).fetchone():  # skip duplicates
        conn.execute("INSERT INTO faq (question, answer, source_url) VALUES (?, ?, ?)", (q, a, src))

# 3. Rewrite answers by id (first person, no class times, stands on its own)
#    e.g. (7, f"Yes! Our beginner classes are a great place to start. See times at {SCHEDULE_URL}")
updates = []
conn.executemany("UPDATE faq SET answer = ? WHERE id = ?", [(text, id_) for id_, text in updates])
conn.commit()

# New questions must be re-embedded; changed answers only need reloading
load_faq()

## Step 4b: The SQL-first router

1. **Q&A (SQL):** stored question similar enough → return the approved answer (free, instant)
2. **Page search (Pinecone):** nothing relevant at all → fallback, no LLM call
3. **Claude:** answers from the top 4 chunks under strict grounding rules, or falls back

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

QA_THRESHOLD = 0.80    # question ↔ stored question: correct matches scored 0.82+, wrong ones ≤ 0.69
RAG_THRESHOLD = 0.50   # question ↔ chunk: junk filter only; Claude decides relevance

FALLBACK = (f"Sorry, I don't have that information. Please contact reception "
            f"on {CONTACT_PHONE} and they'll be happy to help.")

answer_llm = ChatAnthropic(model="claude-sonnet-5-5", max_tokens=2000)  # room for thinking + answer

rag_prompt = ChatPromptTemplate.from_template(f"""
You are the friendly customer support assistant for {SITE_NAME}, {SITE_DESCRIPTION}.

Answer the customer's question using ONLY the website extracts below.

Rules:
- Keep it short: 2-4 sentences maximum, warm and helpful.
- Only include information that directly answers the question. Don't add
  loosely related facts just to say something.
- Speak naturally as the business. Never mention "extracts", "context" or
  "the information I have"; say "our website" or simply answer.
- Mention the phone number at most once.
- Never invent prices, times or policies that are not in the extracts.
- Class times change, so for timetable questions point customers to
  {SCHEDULE_URL} rather than quoting times.
- For health, injury or pregnancy questions, describe what the classes offer
  but never promise health outcomes, and suggest checking with a GP or speaking
  to a teacher first.
- If the extracts only partly answer the question, share what IS relevant,
  say clearly what you can't confirm, and suggest calling {CONTACT_PHONE}.
- Only if the extracts contain nothing relevant at all, reply exactly:
  "{{fallback}}"

Website extracts:
{{context}}

Customer question: {{question}}
""")

def get_text(reply):
    """Return only the answer text, skipping the model's 'thinking' blocks."""
    if isinstance(reply.content, str):
        return reply.content
    return "".join(block.get("text", "") for block in reply.content
                   if isinstance(block, dict) and block.get("type") == "text").strip()

def answer(question):
    # 1. Stored Q&A (SQL) first
    row, score = best_faq_match(question)
    if score >= QA_THRESHOLD:
        return {"route": "Q&A (SQL)", "score": score,
                "answer": row[2], "sources": [row[3]]}

    # 2. Search the page chunks in Pinecone
    results = vectorstore.similarity_search_with_score(question, k=4)
    if not results or results[0][1] < RAG_THRESHOLD:
        return {"route": "No match", "score": results[0][1] if results else 0,
                "answer": FALLBACK, "sources": []}

    # 3. Claude answers from the retrieved chunks
    context = "\n\n".join(f"[{doc.metadata['source']}]\n{doc.page_content}"
                          for doc, _ in results)
    reply = answer_llm.invoke(rag_prompt.format_messages(
        context=context, question=question, fallback=FALLBACK))
    text = get_text(reply)

    if FALLBACK in text:   # Claude found nothing usable: don't show sources
        return {"route": "RAG → no answer in pages", "score": results[0][1],
                "answer": FALLBACK, "sources": []}

    sources = list(dict.fromkeys(doc.metadata["source"] for doc, _ in results))
    return {"route": "RAG (Pinecone + Claude)", "score": results[0][1],
            "answer": text, "sources": sources}

def ask(question):
    r = answer(question)
    print(f"Q: {question}")
    print(f"Route: {r['route']}  (score {r['score']:.3f})")
    print(f"A: {r['answer']}")
    if r["sources"]:
        print("Sources:", ", ".join(r["sources"]))
    print("-" * 60)

ask("how much is a class")             # expect Q&A
ask("can I bring my baby to a class")  # expect RAG
ask("do you sell protein shakes")      # expect fallback

## Test set

21 questions covering every route: Q&A rephrasings, questions only the pages answer, off-topic questions, and awkward phrasing. Compare each route with what you'd expect, then fix and rerun the whole set (fixes can break things that used to work).

In [ ]:
test_questions = [
    # Should hit the Q&A
    "how much does one class cost", "is there a sauna", "where are you",
    "what time does reception open", "what time is reception open on sunday",
    # Should go to the pages and answer
    "what should I bring to a reformer class", "can I do pilates while pregnant",
    "how many people are in a reformer class", "can I hire a room for a workshop",
    "do you do teacher training", "how long after giving birth can I start postnatal yoga",
    "do you run any community or charity classes",
    # Should fall back
    "do you sell protein shakes", "can I bring my dog", "do you have parking",
    "what's the wifi password",
    # Awkward phrasing
    "price?", "sauna how much and when open", "im pregant can i come to yoga",
    "is it good for back pain",
    "what's the best class for a complete beginner who's a bit unfit",
]
for q in test_questions:
    ask(q)

## Step 5: Chat interface

Shows the route and score under each answer for testing; remove that line for a customer-facing version. `launch()` prints a temporary public link: anyone with it can use the bot on your API keys while the session runs, so share it carefully.

In [ ]:
!pip install -q gradio
import gradio as gr

def chat(message, history):
    r = answer(message)
    reply = r["answer"]
    if r["sources"]:
        reply += "\n\n**Sources:** " + ", ".join(r["sources"])
    reply += f"\n\n_route: {r['route']} · score {r['score']:.2f}_"   # for testing; remove for customers
    return reply

gr.ChatInterface(
    fn=chat,
    title=f"{SITE_NAME} Support",
    description="Ask about classes, prices, facilities and more.",
    examples=["How much is a class?", "Can I bring my baby to a class?", "Do you have parking?"],
).launch()

## Save to and restore from Google Drive

Colab's disk is temporary. Save the Q&A database and PDFs to Drive after changing them; after a runtime reset, restore them instead of re-running Steps 1–2. (The Pinecone index is already stored in the cloud.)

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/support_bot"
os.makedirs(SAVE_DIR, exist_ok=True)

shutil.copy("support.db", f"{SAVE_DIR}/support.db")
shutil.copytree("pages", f"{SAVE_DIR}/pages", dirs_exist_ok=True)
print("Saved:", sorted(os.listdir(SAVE_DIR)))

In [ ]:
# Restore after a runtime reset (only copies files that are missing)
from google.colab import drive
import shutil

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/support_bot"

if not os.path.exists("support.db"):
    shutil.copy(f"{SAVE_DIR}/support.db", "support.db")
    shutil.copytree(f"{SAVE_DIR}/pages", "pages", dirs_exist_ok=True)
    print("Restored support.db and PDFs from Drive ✅")
else:
    print("Files already here, nothing to restore")